# Exploration

```{important} AI Disclosure
The written content in this report was drafted by Jenny Lee and refined using Generative AI for clarity and grammar. All ideas, analysis decisions, and interpretations are original. The prompt used throughout was: *"Refine my words."*
```

```{note} Viewing the Context
The written content in this report was drafted by Jenny Lee and refined using Generative AI for clarity and grammar. All ideas, analysis decisions, and interpretations are original. The prompt used throughout was: *"Refine my words."*
```

## Data Collection

In [2]:
import pandas as pd
import numpy as np
import requests
from concurrent.futures import ThreadPoolExecutor, as_completed
import time
import json

## E1
> If you dataset has an unusual structure, describe the steps and
related challenges in extracting the dataset. For example, if your data comes from an API, talk about the
API calls you had to run in order to get the data.

In [3]:
BASE_URL = "https://api.imdbapi.dev/"
response = requests.get(f"{BASE_URL}/titles?types=MOVIE&startYear>=2024")
data = response.json()
df = pd.DataFrame(data["titles"])
pagetoken = data["nextPageToken"]
time.sleep(1)  

for page in range(1, 20):
      try:
            response = requests.get(f"{BASE_URL}/titles?types=MOVIE&startYear=2026&pageToken={pagetoken}", timeout=10)
            
            if response.status_code == 429:
                  wait_time = int(response.headers.get("Retry-After", 5))
                  print(f"Page {page}: Rate limited! Waiting {wait_time}s...")
                  time.sleep(wait_time)
                  continue
            
            response.raise_for_status()
            
            if not response.text:
                  break
            
            data = response.json()
            
            if "titles" not in data or not data["titles"]:
                  break
                  
            df_page = pd.DataFrame(data["titles"])
            df = pd.concat([df, df_page], ignore_index=True)
            
            if "nextPageToken" in data and data["nextPageToken"]:
                  pagetoken = data["nextPageToken"]
            else:
                  break
            
            time.sleep(1)  
      except Exception as e:
            break

print(f"Final shape: {df.shape}")
print(f"Total Number of rows: {len(df)}")
display(df.head())

Final shape: (1000, 10)
Total Number of rows: 1000


,id,type,primaryTitle,originalTitle,primaryImage,startYear,runtimeSeconds,genres,rating,plot
0,tt12042730,movie,Project Hail Mary,Project Hail Mary,{'url': 'https://m.media-amazon.com/images/M/M...,2026.0,9360.0,"[Adventure, Comedy, Sci-Fi]","{'aggregateRating': 8.4, 'voteCount': 178625}",A science teacher wakes up alone on a spaceshi...
1,tt28650488,movie,The Super Mario Galaxy Movie,The Super Mario Galaxy Movie,{'url': 'https://m.media-amazon.com/images/M/M...,2026.0,5880.0,"[Animation, Adventure, Comedy, Family, Fantasy]","{'aggregateRating': 6.5, 'voteCount': 25522}","Mario ventures into space, exploring cosmic wo..."
2,tt32430579,movie,Crime 101,Crime 101,{'url': 'https://m.media-amazon.com/images/M/M...,2026.0,8400.0,"[Crime, Drama, Thriller]","{'aggregateRating': 6.8, 'voteCount': 48321}","An elusive thief, eyeing his final score, enco..."
3,tt33071426,movie,The Drama,The Drama,{'url': 'https://m.media-amazon.com/images/M/M...,2026.0,6300.0,"[Comedy, Drama, Romance]","{'aggregateRating': 7.5, 'voteCount': 17072}",A happily engaged couple is put to the test wh...
4,tt37209937,movie,Pizza Movie,Pizza Movie,{'url': 'https://m.media-amazon.com/images/M/M...,2026.0,5820.0,[Comedy],"{'aggregateRating': 5.8, 'voteCount': 4258}",High college students face an unexpectedly epi...


The greatest challenge faced in extracting data is the usgae limi

In [4]:
def fetch_box_office(ind, row, base_url, delay=0.5):
    title_id = row["id"]
    time.sleep(delay)  
    try:
        response = requests.get(f"{base_url}/titles/{title_id}/boxOffice", timeout=5)
        
        if response.status_code == 429:
            wait_time = int(response.headers.get("Retry-After", 10))
            print(f"Rate limited on {title_id}, waiting {wait_time}s...")
            time.sleep(wait_time)
            return fetch_box_office(ind, row, base_url, delay=2)  #
        
        response.raise_for_status() 
        data = response.json()
        worldwide_gross = data["worldwideGross"]["amount"] if "worldwideGross" in data else None
        production_budget = data["productionBudget"]["amount"] if "productionBudget" in data else None
        return ind, worldwide_gross, production_budget, None
    except requests.Timeout:
        return ind, None, None, f"{title_id}: Timeout"
    except requests.HTTPError as e:
        return ind, None, None, f"{title_id}: HTTP {response.status_code}"
    except json.JSONDecodeError:
        return ind, None, None, f"{title_id}: Invalid JSON response"
    except Exception as e:
        return ind, None, None, f"{title_id}: {type(e).__name__}"

skipped_count = 0
for ind, row in df.iterrows():
    ind_result, worldwide_gross, production_budget, error = fetch_box_office(ind, row, BASE_URL)
    if error:
        skipped_count += 1
    else:
        df.at[ind, "worldwideGross"] = worldwide_gross
        df.at[ind, "productionBudget"] = production_budget

print(f"Skipped {skipped_count} titles due to errors.")
print(f"Box office data added to {len(df) - skipped_count} rows.")

display(df.head(10))

Processed 100/1000 titles...
Processed 200/1000 titles...
Processed 300/1000 titles...
Processed 400/1000 titles...
Processed 500/1000 titles...
Processed 600/1000 titles...
Processed 700/1000 titles...
Processed 800/1000 titles...
Processed 900/1000 titles...
Processed 1000/1000 titles...
Skipped 0 titles due to errors.
Box office data added to 1000 rows.


In [13]:
missing_counts = df.isna().sum()
missing_counts[missing_counts > 0]

primaryImage          4
startYear             5
runtimeSeconds       36
rating               52
plot                  2
worldwideGross      138
productionBudget    232
dtype: int64

In [12]:
missing_df = df[df["worldwideGross"].isna() | df["productionBudget"].isna()]
missing_df["startYear"].value_counts()

startYear
2026.0    98
2025.0    83
2024.0    12
2027.0    10
2023.0     9
2020.0     8
2022.0     7
1973.0     2
2021.0     2
2013.0     2
2017.0     2
2011.0     2
2029.0     1
2018.0     1
1979.0     1
2019.0     1
2000.0     1
1975.0     1
1968.0     1
2007.0     1
2010.0     1
1965.0     1
1985.0     1
1961.0     1
2004.0     1
2002.0     1
1962.0     1
Name: count, dtype: int64